# pymefpp_partition_science

This notebook partitions the science data in raw_data_dir in sets by RA,DEC, FILTER, DATE-OBS, and EXPTIME

In [ ]:
# create a list of science frames
from os import listdir
from os.path import isfile, join
import glob
from astropy.io import fits
import math
import operator
import copy
import datetime
import ephem
import os

def get_phase_on_day(year,month,day):
  """Returns a floating-point number from 0-1. where 0=new, 0.5=full, 1=new"""
  #Ephem stores its date numbers as floating points, which the following uses
  #to conveniently extract the percent time between one new moon and the next
  #This corresponds (somewhat roughly) to the phase of the moon.

  #Use Year, Month, Day as arguments
  date=ephem.Date(datetime.date(year,month,day))

  nnm = ephem.next_new_moon    (date)
  pnm = ephem.previous_new_moon(date)

  lunation=(date-pnm)/(nnm-pnm)

  #Note that there is a ephem.Moon().phase() command, but this returns the
  #percentage of the moon which is illuminated. This is not really what we want.

  return lunation

class Observation:
    
    def __init__(self, filename):

        self.filename = filename
        self.ra = getKeyVal(filename, 'RA')
        self.dec = getKeyVal(filename, 'DEC')
        self.tel_alpha = getKeyVal(filename, 'HIERARCH ESO TEL TARG ALPHA')
        self.tel_delta = getKeyVal(filename, 'HIERARCH ESO TEL TARG DELTA')
        self.target = getKeyVal(filename, 'HIERARCH ESO OBS TARG NAME').replace(' ', '_')
        self.dateobs = getKeyVal(filename, 'DATE-OBS')
        self.OBstart = getKeyVal(filename, 'HIERARCH ESO OBS START')
        self.filter = getKeyVal(filename, 'HIERARCH ESO INS FILT1 NAME')
        self.exptime = getKeyVal(filename, 'EXPTIME')
        self.tplnexp = getKeyVal(filename, 'HIERARCH ESO TPL NEXP')
        self.tplexpno = getKeyVal(filename, 'HIERARCH ESO TPL EXPNO')
        
        # we add the FLI
        yyyy = int(self.dateobs[0:4])
        mm = int(self.dateobs[5:7])
        dd = int(self.dateobs[8:10])
        fli = get_phase_on_day(yyyy,mm,dd)
        if fli <= 0.5:
            fli = fli*2.0
        else:
            fli = 2.0*(1-fli)
        self.fli = fli
        
    
class Partition:

    distanceCriteria = 3.0
        
    def __init__(self):

        self.numberList = []
        self.partList = []                    
        
    def insertObs(self, observation):
        createNewPartition = True
        if len(self.partList) > 0:
            for partition in self.partList:
                if self.getDistance(partition, observation) <= Partition.distanceCriteria and \
                    partition[0].filter == observation.filter:
                    partition.append(observation)
                    createNewPartition = False
                else:
                    pass
        if createNewPartition:
            self.partList.append([observation])
            self.numberList.append(len(self.partList))
            
    def getDistance(self,partition, observation):
        ra = 0.0
        dec = 0.0
        n = 0
        for obs in partition:
            n+=1
            ra+=obs.ra
            dec+=obs.dec
        ra = ra / float(n)
        dec = dec / float(n)
        return math.sqrt((ra-observation.ra)**2 + (dec - observation.dec)**2)

def getKeyVal(filename, key):
    with fits.open(filename) as hdu_list:
        primary_header = hdu_list[0].header
        val = primary_header[key]
        return val



In [ ]:
# we set the data reduction directories
raw_data_dir = '/home/lschmidt/fernando/fits/'

reduced_science_dir = '/home/fselman/omegacam/fernando/reduced/science/'
try:
    os.mkdir(reduced_science_dir)
except:
    pass

In [ ]:

listing = glob.glob(raw_data_dir+'OMEGA.*.fits')

print('%d OmegaCAM frames in %s'%(len(listing), raw_data_dir))

dataPartition = Partition()
science_list = []
i = 0
for filename in listing:
    # open FITS and read headers
    hdu_list = fits.open(filename)
    primary_header = hdu_list[0].header
    dpr_catg = primary_header['HIERARCH ESO DPR CATG']
    hdu_list.close()
    if dpr_catg == 'SCIENCE':
        i += 1
        if i%10:
            print("Reading SCIENCE frame: %d" % i, end='\r')
        observation = Observation(filename)
        science_list.append(observation)
        dataPartition.insertObs(observation)
    

    
print('%d OmegaCAM SCIENCE frames in %s'%(len(science_list), raw_data_dir))   
print("%d partitions"%(len(dataPartition.numberList)))

In [ ]:
# we create directory structure to reduce the data

for partition in dataPartition.partList:
    firstObs = partition[0]
    print(firstObs.target, firstObs.ra, firstObs.dec, firstObs.filter, len(partition))
    obsList = []
    for obs in partition:
        obsList.append([obs.OBstart, obs.tplexpno, obs.tplnexp, obs.fli, obs.filename])
        #rint(obsList)
    newList = copy.deepcopy(sorted(obsList, key = operator.itemgetter(0, 1)))
    #print(newList)
    for anObs in newList:
        subdir_str = firstObs.target+'_'+\
                    ("%010.3f"%firstObs.tel_alpha).replace('.','p')+'_'+\
                    ("%09.1f"%firstObs.tel_delta).replace('.','p').replace('-','m')+'/'+\
                    firstObs.filter+'/'+\
                    anObs[0]
        print(subdir_str)
        try:
            os.makedirs(reduced_science_dir+subdir_str)
        except:
            pass
        
        try:
            os.chdir(reduced_science_dir+subdir_str)
            with open('science.list', 'a') as f:
                print(anObs[4], file=f)
            os.chdir(reduced_science_dir)
        except:
            raise Exception('could not change directory to: '+reduced_science_dir+subdir_str)
  
        #print('  ', anObs[0], anObs[1], anObs[2], "%4.2f"%anObs[3], anObs[4], getKeyVal(anObs[4], 'HIERARCH ESO INS FILT1 NAME'))